# Step 7 — Label Creation + Hard/Easy Negative Mining

**What changed vs. the version you gave me, and why (most severe first):**

1. **Reading `GROUND_TRUTH` as a plain CSV is a real data-corruption risk, not a style nitpick.** The
   challenge spec is explicit that these files are TAB-separated *specifically because*
   `matched_entity_ids` contains commas - and warns that reading a `.tsv` without `sep="\t"` silently
   produces garbage rather than an error. Your example path is `train_truth.csv`, which raises the
   question of whether that file is genuinely comma-separated (in which case the commas *inside*
   `matched_entity_ids` would get split into extra columns unless properly quoted) or just a `.tsv` that
   was renamed. Rather than guessing, the loader now tries tab first, falls back to comma, and validates
   that both expected columns actually exist either way - so it fails loudly instead of silently
   corrupting labels.
2. **No blocking-recall diagnostic** - this is exactly the step where it can and should be computed: this
   is the first point where you have both the exhaustive ground-truth positive pairs *and* the actual
   candidate/feature set they're being merged against. A true match that blocking never generated as a
   candidate is gone forever, however good the model is - and the original script had no visibility into
   how many of those there were. Added a printed recall-ceiling check against the ~97-98% target flagged
   earlier in this project.
3. **Hard-negative mining only looked at `bge_cosine`.** A candidate found only by TF-IDF (word or char)
   with `bge_cosine = NaN` would evaluate `NaN >= 0.80` as `False` and get silently bucketed as "easy"
   regardless of how high its TF-IDF score actually was. Switched to the `combined_score` field Step 6
   already computes (max across all three blockers), with a fallback if it's missing.
4. **No easy negatives at all.** Keeping *only* hard negatives + positives means the model never sees what
   a clearly-wrong, low-similarity pair looks like during training - exactly the risk flagged earlier in
   this project's compute-optimization discussion. At real inference time the model scores the *entire*
   candidate set, most of which is nowhere near the 0.80 hard-negative threshold; training on hard
   negatives alone risks miscalibrated behavior across that lower range it never saw. Added a small random
   easy-negative sample, and an optional cap on hard negatives (both configurable, both off by default
   capping-wise if you set them to `None`).
5. **A duplicate-key merge risk in two places**: (a) `labels` isn't deduplicated before the merge - if
   ground truth ever lists the same candidate twice for one entity, that would silently multiply rows on
   the `features` side of the left-join; (b) no check that `features` itself has no duplicate
   `(entity_id, candidate_id)` pairs going in. Both now asserted/deduplicated explicitly.
6. **A latent edge-case bug**: the original only checked `pd.isna(matched_entity_ids)` to skip
   no-match rows - if that field were ever an *empty string* rather than true NaN, `"".split(",")`
   returns `['']`, which would silently create a spurious positive-labeled pair with `candidate_id=""`.
   Now excludes both NaN and empty/whitespace-only strings.
7. **`iterrows()` over the ground-truth file**, replaced with a vectorized `str.split` + `explode` -
   ground truth is one row per Source 1 entity (not per pair), so this was less severe than the earlier
   per-pair steps, but still worth fixing for consistency and because Source 1 alone is on the order of
   millions of rows.

Kept in one cell, matching the convention of the other `stepN_*.py` scripts.

In [ ]:
# ==========================================================
# PUT YOUR PATHS HERE
# ==========================================================

TRAIN_FEATURES = r"PASTE_MERGED_TRAIN_FEATURES.parquet"
GROUND_TRUTH = r"PASTE_GROUND_TRUTH.csv"
OUTPUT_FOLDER = r"PASTE_OUTPUT_FOLDER"

# ==========================================================
# NEGATIVE SAMPLING CONFIG
# ==========================================================

HARD_NEGATIVE_THRESHOLD = 0.80
MAX_HARD_NEG_PER_POSITIVE = 10   # cap hard negatives at ~10x positives; set to None for no cap
EASY_NEGATIVE_PER_POSITIVE = 2   # small random sample of low-similarity negatives, for score-range calibration
RANDOM_STATE = 42

# ==========================================================

from pathlib import Path
import pandas as pd

OUT = Path(OUTPUT_FOLDER)
OUT.mkdir(parents=True, exist_ok=True)

print("Loading feature dataset...")
features = pd.read_parquet(TRAIN_FEATURES)

assert not features.duplicated(subset=["entity_id", "candidate_id"]).any(), (
    "Duplicate (entity_id, candidate_id) pairs in the feature file - check Step 5/6 output."
)


# ----------------------------------------------------------
# Load ground truth - see point 1 above on why the separator matters here.
# ----------------------------------------------------------
def read_ground_truth(path):
    for sep, sep_name in [("\t", "tab"), (",", "comma")]:
        try:
            df = pd.read_csv(path, sep=sep)
        except Exception:
            continue
        if {"source1_entity_id", "matched_entity_ids"}.issubset(df.columns):
            print(f"  Ground truth read as {sep_name}-separated ({len(df):,} rows).")
            return df
    raise ValueError(
        f"Could not find 'source1_entity_id' and 'matched_entity_ids' columns in {path} "
        f"as either tab- or comma-separated. The challenge format is TAB-separated "
        f"specifically because matched_entity_ids contains commas - check this file "
        f"wasn't accidentally saved/converted as a plain comma-separated CSV, which "
        f"would silently split that column apart."
    )


print("Loading ground truth...")
truth = read_ground_truth(GROUND_TRUTH)

# ----------------------------------------------------------
# Convert truth into (entity_id, candidate_id) - vectorized instead of
# iterrows(), and excludes both NaN and empty-string matched_entity_ids.
# ----------------------------------------------------------
has_matches = truth["matched_entity_ids"].notna() & (truth["matched_entity_ids"].astype(str).str.strip() != "")
truth_matched = truth[has_matches].copy()
truth_matched["matched_entity_ids"] = truth_matched["matched_entity_ids"].astype(str).str.split(",")

labels = truth_matched.explode("matched_entity_ids")
labels["candidate_id"] = labels["matched_entity_ids"].str.strip()
labels = labels.rename(columns={"source1_entity_id": "entity_id"})[["entity_id", "candidate_id"]]
labels["label"] = 1
labels = labels.drop_duplicates(subset=["entity_id", "candidate_id"])  # guards the merge below against fan-out

# ----------------------------------------------------------
# Merge labels
# ----------------------------------------------------------
train = features.merge(labels, on=["entity_id", "candidate_id"], how="left")
train["label"] = train["label"].fillna(0).astype("int8")

# ----------------------------------------------------------
# Blocking recall ceiling - check this before trusting anything downstream.
# ----------------------------------------------------------
total_gt_pairs = len(labels)
recovered_pairs = int(train["label"].sum())
recall_ceiling = recovered_pairs / total_gt_pairs if total_gt_pairs else float("nan")
print(f"Blocking recall ceiling (train): {recovered_pairs:,}/{total_gt_pairs:,} = {recall_ceiling:.2%}")
if total_gt_pairs and recall_ceiling < 0.97:
    print("  WARNING: below the ~97-98% recall-ceiling target flagged earlier in this "
          "project - worth revisiting Step 3/5 blocking before spending time on modeling.")

# ----------------------------------------------------------
# Hard / easy negative mining
# Keep:
#   All positives
#   Hard negatives: label==0 with a high combined similarity score
#     (any blocker, not just bge_cosine - see point 3 above)
#   A small random sample of easy negatives, so the model sees the low
#     end of the score range it'll actually face at inference time
# ----------------------------------------------------------
if "combined_score" in train.columns:
    hardness_score = train["combined_score"]
else:
    hardness_score = train[["bge_cosine", "tfidf_word_score", "tfidf_char_score"]].max(axis=1)

positive = train[train.label == 1]
n_pos = len(positive)

hard_negative_pool = train[(train.label == 0) & (hardness_score >= HARD_NEGATIVE_THRESHOLD)]
if MAX_HARD_NEG_PER_POSITIVE is not None and n_pos > 0:
    hard_cap = n_pos * MAX_HARD_NEG_PER_POSITIVE
    if len(hard_negative_pool) > hard_cap:
        hard_negative_pool = hard_negative_pool.sample(n=hard_cap, random_state=RANDOM_STATE)

easy_negative_pool = train[(train.label == 0) & (hardness_score < HARD_NEGATIVE_THRESHOLD)]
easy_target = min(len(easy_negative_pool), n_pos * EASY_NEGATIVE_PER_POSITIVE) if EASY_NEGATIVE_PER_POSITIVE else 0
easy_negative_sample = easy_negative_pool.sample(n=easy_target, random_state=RANDOM_STATE)

final_train = pd.concat([positive, hard_negative_pool, easy_negative_sample], ignore_index=True)

# Shuffle
final_train = final_train.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------
final_train.to_parquet(OUT / "final_train_dataset.parquet", index=False, compression="zstd")

print("=" * 50)
print("Dataset Created Successfully")
print("=" * 50)
print(f"Positives      : {n_pos:,}")
print(f"Hard negatives : {len(hard_negative_pool):,}")
print(f"Easy negatives : {len(easy_negative_sample):,}")
print(f"Final rows     : {len(final_train):,}")